# Fine-tune East Frisian (Oostfräisk) TTS with Piper

This notebook fine-tunes a VITS model from [Piper](https://github.com/rhasspy/piper) on East Frisian Low Saxon data.

**Two training modes available:**

| Mode | Phonemizer | Pros | Cons |
|------|-----------|------|------|
| **`grapheme`** | None — raw characters | Simple, no preprocessing hack, keeps original spelling | Needs more data to learn pronunciation patterns |
| **`espeak`** | espeak-ng (German) | Leverages German pronunciation knowledge | Requires text preprocessing (ğ→ch, ó→oa, etc.) |

Set your preferred mode in the cell below.

**Requirements:** Google Colab with A100 GPU

In [ ]:
# =============================================
# CONFIGURATION — set your preferred mode here
# =============================================

PHONEME_MODE = "grapheme"   # "grapheme" or "espeak"

# Training hyperparameters
BATCH_SIZE = 16
MAX_EPOCHS = 2000
QUALITY = "medium"          # "medium" (22050 Hz) or "high" (22050 Hz, larger model)

print(f"Mode: {PHONEME_MODE}")
print(f"Batch size: {BATCH_SIZE}, Max epochs: {MAX_EPOCHS}, Quality: {QUALITY}")

# 1. Install Dependencies

In [ ]:
# espeak-ng is needed by piper-train internally (even in grapheme mode)
!apt-get install -y espeak-ng

# Install Piper training tools
!pip install -U pip
!git clone --depth 1 https://github.com/rhasspy/piper.git /tmp/piper
!cd /tmp/piper/src/python && pip install -e .

# Install piper-tts for inference testing
!pip install piper-tts

# HuggingFace Hub for checkpoint download
!pip install huggingface_hub

# 2. Clone Repository & Setup

In [ ]:
!git clone https://VanModers:@github.com/VanModers/oostfraeisk_text_to_speech
%cd oostfraeisk_text_to_speech
!git pull

# 3. GPU Optimizations

In [ ]:
import torch

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"TF32 enabled: {torch.backends.cuda.matmul.allow_tf32}")

# 4. Download Pretrained German Checkpoint

Fine-tune from the German **Thorsten** voice (medium quality = VITS @ 22050 Hz).

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_tree
import os

repo_id = "rhasspy/piper-checkpoints"

# Find the German Thorsten medium checkpoint
all_files = [
    f.rfilename for f in list_repo_tree(repo_id, repo_type="dataset")
    if hasattr(f, 'rfilename')
    and "thorsten" in f.rfilename
    and "medium" in f.rfilename
    and f.rfilename.endswith(".ckpt")
]
print("Available Thorsten medium checkpoints:")
for f in all_files:
    print(f"  {f}")

if all_files:
    pretrained_ckpt = hf_hub_download(
        repo_id=repo_id, filename=all_files[0], repo_type="dataset"
    )
else:
    pretrained_ckpt = hf_hub_download(
        repo_id=repo_id,
        filename="de/de_DE/thorsten/medium/epoch=2164-step=1355540.ckpt",
        repo_type="dataset"
    )

print(f"\nDownloaded: {pretrained_ckpt}")

# 5. Preprocess Dataset

This step differs depending on the selected mode:

- **`grapheme`**: Custom preprocessing — builds a character-to-ID map from the actual text, caches audio. No phonemizer.
- **`espeak`**: Converts East Frisian orthography to German-compatible forms, then uses Piper's built-in espeak-ng preprocessing.

### Why custom preprocessing for grapheme mode?

Piper's built-in `--phoneme-type text` only has a character-to-ID map for Ukrainian in its source code. For any other language, you'd get a KeyError. So we do the same thing manually: build a character-to-ID map from the dataset, convert each utterance to IDs, and cache the audio — producing the exact same output format Piper's trainer expects.

In [ ]:
# East Frisian → German phoneme mapping (only used in espeak mode)
custom_phoneme_map = {
    # Complex diphthongs / triphthongs (longest first)
    "öye": "öije",    # /œyə/ - göyen (gießen)
    "ööe": "ööö",    # extra-long ö
    "óóej": "ooai",  # /ɒ:ɛɪ/ - dóóejt (Tat)
    "âau": "aau",    # /a:ʊ/ - brâau
    "âaj": "aai",    # /a:ɪ/ - drâajen (drehen)
    "êer": "eer",    # /e:r/ - fêert (fährt)
    "êel": "eel",    # /e:l/ - fêelen (fühlen)

    # Circumflex (extra-long) vowels
    "ââ": "aa", "êê": "ee", "îî": "ii", "ôô": "oo", "ûû": "uu",
    "âa": "aa", "êe": "ee", "îi": "ii", "ôo": "oo", "ûu": "uu",

    # East Frisian specific vowels
    "óój": "oai", "óó": "oa", "ó": "oa",

    # ö-diphthongs
    "öy": "öi", "öej": "ööi", "öj": "öi",

    # ä-diphthongs
    "äie": "ääi", "äej": "ääi", "äj": "äi", "äi": "äi",

    # Basic diphthongs
    "ooj": "ooi", "oi": "oi", "ei": "ei",
    "aaj": "aai", "ai": "ai", "aau": "aau", "au": "au",

    # Consonants
    "ğ": "ch", "tj": "tsch",
}

_sorted_keys = sorted(custom_phoneme_map.keys(), key=len, reverse=True)

def preprocess_east_frisian(text: str) -> str:
    """Convert East Frisian orthography to German-compatible forms."""
    result = text
    for key in _sorted_keys:
        result = result.replace(key, custom_phoneme_map[key])
    return result

# Quick test
for t in ["Suldóót", "fandóóeğ", "drâajen"]:
    print(f"  {t} → {preprocess_east_frisian(t)}")

In [ ]:
import csv
import json
import shutil
from pathlib import Path
from collections import Counter

DATASET_DIR = Path("data/oostfraeisk")
TRAINING_DIR = Path("piper_training")
SAMPLE_RATE = 22050

metadata_path = DATASET_DIR / "metadata.csv"

if PHONEME_MODE == "espeak":
    # ── ESPEAK MODE ─────────────────────────────────────
    # Preprocess text: East Frisian → German-compatible
    # Then use piper_train.preprocess with espeak-ng (de)

    backup_path = metadata_path.with_suffix('.csv.original')
    if not backup_path.exists():
        shutil.copy(metadata_path, backup_path)
        print(f"Backed up original to {backup_path}")
    else:
        shutil.copy(backup_path, metadata_path)
        print(f"Restored original from {backup_path}")

    with open(metadata_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    processed_lines = []
    for line in lines:
        parts = line.strip().split('|')
        if len(parts) >= 2:
            filename = parts[0]
            text = parts[1]
            processed = preprocess_east_frisian(text)
            if len(parts) == 3:
                processed_lines.append(f"{filename}|{processed}|{processed}\n")
            else:
                processed_lines.append(f"{filename}|{processed}\n")
        else:
            processed_lines.append(line)

    with open(metadata_path, 'w', encoding='utf-8') as f:
        f.writelines(processed_lines)

    print(f"Preprocessed {len(processed_lines)} lines for espeak mode")
    print(f"Example: {lines[0].strip()}")
    print(f"       → {processed_lines[0].strip()}")

elif PHONEME_MODE == "grapheme":
    print("Grapheme mode — no text preprocessing needed.")
    print("Original East Frisian text will be used as-is.")

else:
    raise ValueError(f"Unknown PHONEME_MODE: {PHONEME_MODE}")

In [ ]:
if PHONEME_MODE == "espeak":
    # ── ESPEAK: use Piper's built-in preprocessor ───────
    import subprocess
    subprocess.run([
        "python", "-m", "piper_train.preprocess",
        "--language", "de",
        "--input-dir", str(DATASET_DIR),
        "--output-dir", str(TRAINING_DIR),
        "--dataset-format", "ljspeech",
        "--single-speaker",
        "--sample-rate", str(SAMPLE_RATE),
    ], check=True)
    print("\nEspeak preprocessing complete!")

elif PHONEME_MODE == "grapheme":
    # ── GRAPHEME: custom preprocessing ──────────────────
    # We create our own config.json + dataset.jsonl with a custom
    # character-to-ID map covering ALL East Frisian characters.

    from piper_train.norm_audio import cache_norm_audio, make_silence_detector

    TRAINING_DIR.mkdir(parents=True, exist_ok=True)
    cache_dir = TRAINING_DIR / "cache" / str(SAMPLE_RATE)
    cache_dir.mkdir(parents=True, exist_ok=True)

    # 1. Read all utterances and collect unique characters
    utterances = []  # (filename, text_orig, text_lower, wav_path)
    all_chars = set()

    wav_dir = DATASET_DIR / "wavs"
    if not wav_dir.is_dir():
        wav_dir = DATASET_DIR / "wav"

    with open(metadata_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='|')
        for row in reader:
            filename = row[0]
            text = row[-1]  # last column is always the text
            text_lower = text.casefold()

            # Find WAV file
            wav_path = wav_dir / f"{filename}.wav"
            if not wav_path.exists():
                wav_path = wav_dir / filename
            if not wav_path.exists():
                wav_path = DATASET_DIR / f"{filename}.wav"
            if not wav_path.exists():
                wav_path = DATASET_DIR / filename

            if wav_path.exists() and wav_path.stat().st_size > 0:
                utterances.append((filename, text, text_lower, wav_path))
                all_chars.update(text_lower)
            else:
                print(f"WARNING: Missing {filename}")

    print(f"Found {len(utterances)} utterances")
    print(f"Unique characters ({len(all_chars)}): {''.join(sorted(all_chars))}")

    # 2. Build phoneme_id_map (character → list of IDs)
    # Reserved: 0=pad(_), 1=bos(^), 2=eos($), 3=space( )
    phoneme_id_map = {
        "_": [0],   # pad
        "^": [1],   # beginning of sentence
        "$": [2],   # end of sentence
        " ": [3],   # word separator
    }

    next_id = 4
    for char in sorted(all_chars):
        if char == " ":
            continue
        phoneme_id_map[char] = [next_id]
        next_id += 1

    num_symbols = max(next_id, 256)  # Piper expects at least 256

    print(f"\nPhoneme ID map ({len(phoneme_id_map)} entries, {num_symbols} symbol slots):")
    for char, ids in list(phoneme_id_map.items())[:20]:
        print(f"  {repr(char):8s} → {ids}")
    if len(phoneme_id_map) > 20:
        print(f"  ... and {len(phoneme_id_map) - 20} more")

    # 3. Write config.json
    config = {
        "dataset": "oostfraeisk",
        "audio": {
            "sample_rate": SAMPLE_RATE,
            "quality": QUALITY,
        },
        "espeak": {
            "voice": "de",  # not used, but Piper expects the field
        },
        "language": {
            "code": "de",
        },
        "inference": {
            "noise_scale": 0.667,
            "length_scale": 1.0,
            "noise_w": 0.8,
        },
        "phoneme_type": "text",
        "phoneme_map": {},
        "phoneme_id_map": phoneme_id_map,
        "num_symbols": num_symbols,
        "num_speakers": 1,
        "speaker_id_map": {},
    }

    with open(TRAINING_DIR / "config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=4)
    print(f"\nWrote config.json (phoneme_type=text, {num_symbols} symbols)")

    # 4. Process utterances: chars→IDs + cache audio
    silence_detector = make_silence_detector()

    print(f"\nProcessing {len(utterances)} utterances...")
    with open(TRAINING_DIR / "dataset.jsonl", "w", encoding="utf-8") as dataset_file:
        for i, (filename, text_orig, text_lower, wav_path) in enumerate(utterances):
            # Each character is a "phoneme"
            phonemes = list(text_lower)

            # Build ID sequence with interleaved padding: BOS pad c1 pad c2 pad ... EOS
            phoneme_ids = [1, 0]  # BOS, pad
            for char in text_lower:
                if char in phoneme_id_map:
                    phoneme_ids.extend(phoneme_id_map[char])
                    phoneme_ids.append(0)  # pad
            phoneme_ids.append(2)  # EOS

            # Cache normalized audio
            audio_norm_path, audio_spec_path = cache_norm_audio(
                wav_path, cache_dir, silence_detector, SAMPLE_RATE
            )

            entry = {
                "text": text_orig,
                "audio_path": str(wav_path.absolute()),
                "phonemes": phonemes,
                "phoneme_ids": phoneme_ids,
                "audio_norm_path": str(audio_norm_path),
                "audio_spec_path": str(audio_spec_path),
            }
            json.dump(entry, dataset_file, ensure_ascii=False)
            print("", file=dataset_file)  # newline

            if (i + 1) % 100 == 0:
                print(f"  {i + 1}/{len(utterances)}...")

    print(f"\nDone! Wrote {len(utterances)} utterances to dataset.jsonl")

In [ ]:
# Verify preprocessing output
import json

with open(TRAINING_DIR / "config.json") as f:
    cfg = json.load(f)

print("Training config:")
print(f"  Phoneme type:  {cfg.get('phoneme_type')}")
print(f"  Sample rate:   {cfg.get('audio', {}).get('sample_rate')}")
print(f"  Num symbols:   {cfg.get('num_symbols')}")
print(f"  ID map entries: {len(cfg.get('phoneme_id_map', {}))}")

with open(TRAINING_DIR / "dataset.jsonl") as f:
    first = json.loads(f.readline())
print(f"\nFirst utterance:")
print(f"  Text:        {first['text']}")
print(f"  Phonemes:    {first['phonemes'][:30]}...")
print(f"  Phoneme IDs: {first['phoneme_ids'][:30]}...")

# 6. Train the Model

Fine-tune from the pretrained German Thorsten checkpoint.

**Tips:**
- Monitor `loss_disc_all` in tensorboard — model is done when it plateaus
- espeak mode: ~1000 extra epochs usually sufficient for fine-tuning
- grapheme mode: may need more epochs (~2000+) since the model must learn pronunciation from scratch
- Adjust `--batch-size` if you run out of VRAM

In [ ]:
cmd = " ".join([
    "python -m piper_train",
    f"--dataset-dir {TRAINING_DIR}",
    "--accelerator gpu --devices 1",
    f"--batch-size {BATCH_SIZE}",
    "--validation-split 0.05",
    "--num-test-examples 5",
    f"--max-epochs {MAX_EPOCHS}",
    f'--resume_from_single_speaker_checkpoint "{pretrained_ckpt}"',
    "--checkpoint-epochs 5",
    "--precision 32",
    f"--quality {QUALITY}",
])

print("Training command:")
print(cmd)

In [ ]:
# Run training (this will take a while)
!python -m piper_train \
    --dataset-dir {str(TRAINING_DIR)} \
    --accelerator gpu \
    --devices 1 \
    --batch-size {BATCH_SIZE} \
    --validation-split 0.05 \
    --num-test-examples 5 \
    --max-epochs {MAX_EPOCHS} \
    --resume_from_single_speaker_checkpoint "{pretrained_ckpt}" \
    --checkpoint-epochs 5 \
    --precision 32 \
    --quality {QUALITY}

# 7. Find Best Checkpoint

In [ ]:
import glob

ckpts = sorted(glob.glob(str(TRAINING_DIR / "lightning_logs/version_*/checkpoints/*.ckpt")))

print("Available checkpoints:")
for ckpt in ckpts[-10:]:
    print(f"  {ckpt}")

if ckpts:
    best_ckpt = ckpts[-1]
    print(f"\nUsing latest: {best_ckpt}")
else:
    print("No checkpoints found!")

# 8. Export to ONNX

Creates `model.onnx` for fast CPU inference. The config is copied alongside as `model.onnx.json`.

In [ ]:
import os

!mkdir -p model_piper

!python -m piper_train.export_onnx \
    "{best_ckpt}" \
    model_piper/oostfraeisk.onnx

# Copy training config as ONNX config
!cp {str(TRAINING_DIR)}/config.json model_piper/oostfraeisk.onnx.json

onnx_path = "model_piper/oostfraeisk.onnx"
if os.path.exists(onnx_path):
    size_mb = os.path.getsize(onnx_path) / 1e6
    print(f"\nExported: {onnx_path} ({size_mb:.1f} MB)")
else:
    print("Export failed!")

# 9. Test Inference

In [ ]:
from piper import PiperVoice
import wave

voice = PiperVoice.load("model_piper/oostfraeisk.onnx")
print(f"Model loaded! Sample rate: {voice.config.sample_rate}")
print(f"Phoneme type: {voice.config.phoneme_type}")

In [ ]:
test_sentences = [
    "Moin, woo gaajt 't dii?",
    "Denkent jii, dat ik disser sats gaud uutprooten dau?",
    "Hest duu däi süen fandóóeğ al säin?",
    "Däi oorsprungelk tóól fan däi Fräisen tüsken Laauwers un Wäiser was dat Olfräisk.",
]

for i, text in enumerate(test_sentences):
    # In espeak mode, preprocess; in grapheme mode, use as-is
    synth_text = preprocess_east_frisian(text) if PHONEME_MODE == "espeak" else text
    out_path = f"test_piper_{i}.wav"
    with wave.open(out_path, "w") as wav_file:
        voice.synthesize(synth_text, wav_file)
    print(f"[{i}] {text}")
    if PHONEME_MODE == "espeak":
        print(f"    → {synth_text}")
    print()

In [ ]:
import IPython
IPython.display.Audio("test_piper_0.wav")

In [ ]:
IPython.display.Audio("test_piper_1.wav")

In [ ]:
IPython.display.Audio("test_piper_2.wav")

In [ ]:
IPython.display.Audio("test_piper_3.wav")

# 10. Push to Git

In [ ]:
# Restore original metadata.csv if we modified it (espeak mode)
import shutil
from pathlib import Path

backup_path = Path("data/oostfraeisk/metadata.csv.original")
if backup_path.exists():
    shutil.copy(backup_path, "data/oostfraeisk/metadata.csv")
    print("Restored original metadata.csv")

In [ ]:
# Clean up training artifacts
!rm -rf piper_training/lightning_logs
!rm -rf piper_training/cache
print("Cleaned training artifacts")

In [ ]:
!git config --global user.email "programmingstudios227@gmail.com"
!git config --global user.name "Tido Specht"
!git add model_piper/
!git add -A
!git commit -m "Piper model treenäärt"
!git push

# 11. Deploy to HuggingFace Space

The HF Space needs:
- `model.onnx` + `model.onnx.json` — the exported model
- `app.py` — use `app_piper.py` from this repo
- `requirements.txt`: `piper-tts`, `gradio`

**Important:** If you trained in **grapheme mode**, set `PHONEME_MODE = "grapheme"` in `app_piper.py` — the East Frisian→German preprocessing will be skipped since the model understands native letters.

In [ ]:
!pip install huggingface_hub ipywidgets

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
%cd ..
!git clone https://huggingface.co/spaces/VanModers114/East_Frisian_TTS

In [ ]:
# Copy model + app to HF Space
!cp oostfraeisk_text_to_speech/model_piper/oostfraeisk.onnx East_Frisian_TTS/model.onnx
!cp oostfraeisk_text_to_speech/model_piper/oostfraeisk.onnx.json East_Frisian_TTS/model.onnx.json
!cp oostfraeisk_text_to_speech/app_piper.py East_Frisian_TTS/app.py

# Create requirements.txt
!echo "piper-tts" > East_Frisian_TTS/requirements.txt
!echo "gradio" >> East_Frisian_TTS/requirements.txt

In [ ]:
%cd East_Frisian_TTS
!git add -A
!git commit -m "Piper ONNX model"
!git push